<a href="https://colab.research.google.com/github/robertbarcik/vector_databases-tutorial/blob/main/1.%20ChromaDB/2_Semantic_Search_and_RAG_with_ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Semantic Search and RAG with ChromaDB

The store exists. Six cybersecurity articles sit in a Chroma collection with their vectors and metadata. Now we do what a vector database is for: ask a question in plain words, get the closest documents back, and hand them to a language model so the answer is grounded in *our* text rather than in whatever the model remembers.

Three parts: **semantic search** with `query()`, the same search **combined with metadata filters**, and a **RAG pipeline** that is the previous course's hand-built version with Chroma doing the retrieval. At the end, an optional section runs the whole thing with a local open model, so no text ever leaves your machine.


## Setup

The generation step calls OpenAI (`gpt-5.6-luna`, cents per notebook). The key lookup goes Colab secret, then environment variable, then a prompt.


In [1]:
%pip install -q chromadb==1.5.5 openai==2.28.0   # Colab installs here; locally, `pip install -r requirements.txt` in the course folder already covers it

import os, json, pathlib, urllib.request
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

client = OpenAI(api_key=api_key)
MODEL = "gpt-5.6-luna"   # small, cheap model of the current generation

RAW = "https://raw.githubusercontent.com/robertbarcik/vector_databases-tutorial/main/1.%20ChromaDB/"
if not pathlib.Path("cyber_documents.json").exists():
    urllib.request.urlretrieve(RAW + "cyber_documents.json", "cyber_documents.json")

os.environ["ANONYMIZED_TELEMETRY"] = "False"
import chromadb
from chromadb.utils import embedding_functions

print("Ready. Model:", MODEL)


Note: you may need to restart the kernel to use updated packages.


Ready. Model: gpt-5.6-luna


## Opening the store

If you ran the first notebook in this folder, the `chroma_db/` directory is here and the collection opens in one line. On Colab (or a fresh machine) the folder does not exist, so the cell rebuilds the collection from the JSON file. Either way you end up with the same six documents and the same local embedding function, which matters: **queries must be embedded by the same model as the stored documents**.


In [2]:
embed_local = embedding_functions.DefaultEmbeddingFunction()
chroma = chromadb.PersistentClient(path="./chroma_db")

collection = chroma.get_or_create_collection(
    name="cyber_docs", embedding_function=embed_local, configuration={"hnsw": {"space": "cosine"}}
)

if collection.count() == 0:                      # fresh machine: rebuild from the data file
    documents = json.load(open("cyber_documents.json"))
    collection.add(
        ids=[d["id"] for d in documents],
        documents=[d["text"] for d in documents],
        metadatas=[d["metadata"] for d in documents],
    )
    print("rebuilt the collection from cyber_documents.json")

print("documents in 'cyber_docs':", collection.count())


documents in 'cyber_docs': 6


## 1. Semantic search with `query()`

In the previous course you embedded the question yourself and ran a cosine loop over a DataFrame. `collection.query()` does both steps: it embeds the query text with the collection's embedding function, compares it with every stored vector, and returns the closest `n_results`. Before you run it: which of the six articles would *you* pick for this question?


In [3]:
question = "What are the foundational principles and technologies used to secure modern internet traffic?"

results = collection.query(query_texts=[question], n_results=2)

for rank, (doc_id, distance, text) in enumerate(zip(results["ids"][0], results["distances"][0], results["documents"][0]), start=1):
    print(f"{rank}. {doc_id}   distance {distance:.3f}")
    print("   ", text[:160], "...\n")


1. document_6   distance 0.548
    A Virtual Private Network (VPN) creates a secure, encrypted tunnel between your device and the internet. When you connect to a VPN, all your internet traffic is ...

2. document_1   distance 0.576
    Public-key cryptography, also known as asymmetric cryptography, represents a monumental paradigm shift from its predecessor, symmetric cryptography. The fundame ...



### 🔍 What just happened?

Every field in the result is a list of lists, one inner list per query text, which is why we index `[0]`. The **distance** is cosine distance: 0 means identical direction, around 1 means unrelated, and the smaller the number the better the match. The VPN article won by a nose over public-key cryptography: both are about encrypting data in transit, one through a tunnel, the other through TLS and HTTPS. Notice that neither text contains the words "foundational principles"; the match is on meaning, not on shared words. That is the difference between semantic search and keyword search.

### 🎯 Mini-task

Ask the same question in different words ("How do we protect data on its way through the internet?") and check whether the winner stays the same. Then ask something the collection does not cover ("How do I bake bread?") and look at the distances.


In [4]:
# YOUR CODE HERE


## 2. Search plus filters

Real systems rarely search everything. A support bot searches only the customer's own contracts; a course assistant searches only beginner material. The `where` filter you met in the first notebook plugs straight into `query()`: Chroma keeps the documents that pass the filter and ranks only those by meaning.


In [5]:
question = "How do we make sure the person logging in is who they claim to be?"

results = collection.query(
    query_texts=[question],
    n_results=3,
    where={"difficulty": "beginner"},
    include=["metadatas", "distances"],
)

print("beginner documents only:")
for doc_id, meta, distance in zip(results["ids"][0], results["metadatas"][0], results["distances"][0]):
    print(f"  {doc_id:<11} {meta['category']:<20} distance {distance:.3f}")


beginner documents only:
  document_5  authentication       distance 0.652
  document_4  social-engineering   distance 0.787
  document_6  network-security     distance 0.874


Metadata is not the only filter. `where_document` looks inside the text itself, which is handy for names, codes and product IDs that embeddings tend to blur.


In [6]:
results = collection.query(
    query_texts=["attacks on industrial systems"],
    n_results=3,
    where_document={"$contains": "certificate"},   # only documents whose text contains this word
    include=["distances"],
)

print("documents mentioning 'certificate', ranked by meaning:")
for doc_id, distance in zip(results["ids"][0], results["distances"][0]):
    print(f"  {doc_id}  distance {distance:.3f}")


documents mentioning 'certificate', ranked by meaning:
  document_2  distance 0.598
  document_1  distance 0.768


### 🔍 What just happened?

The first query never saw the advanced and intermediate articles, so the MFA article won among the three beginner ones. The second query mixed a keyword condition with a semantic ranking: Stuxnet came first because it was signed with stolen certificates and is about an industrial attack; the cryptography article mentions certificates too but is further from the question. This combination of **filter first, rank by meaning second** is the everyday shape of production retrieval.

### 🎯 Mini-task

Search for "modern security approaches" but only in documents from 2010 or later (`where={"year": {"$gte": 2010}}`). Which three come back, and in which order?


In [7]:
# YOUR CODE HERE


## 3. How many documents to retrieve?

`n_results` is the one knob you will keep tuning. Too few and the answer may miss a document that had the missing half of it; too many and you pay for tokens the model does not need, and the extra text can pull the answer off course. Let's watch both the distances and the size of the context grow.


In [8]:
question = "What are the main security threats?"

for n in [1, 3, 5]:
    results = collection.query(query_texts=[question], n_results=n, include=["documents", "distances"])
    words = sum(len(doc.split()) for doc in results["documents"][0])
    print(f"n_results={n}: {results['ids'][0]}")
    print(f"   distances {[round(d, 3) for d in results['distances'][0]]}   context ≈ {words} words\n")


n_results=1: ['document_4']
   distances [0.518]   context ≈ 396 words

n_results=3: ['document_4', 'document_3', 'document_5']
   distances [0.518, 0.588, 0.625]   context ≈ 992 words



n_results=5: ['document_4', 'document_3', 'document_5', 'document_1', 'document_2']
   distances [0.518, 0.588, 0.625, 0.662, 0.675]   context ≈ 1771 words



### 🔍 What just happened?

The best match is always the same; more results just append weaker ones, and the distances tell you how much weaker. A practical recipe: start at 3 to 5 for article-sized documents, more for short chunks, and drop results whose distance is clearly worse than the top one. The function below does exactly that with a fixed threshold, the simplest form of "retrieve fewer when the store has little to say".


In [9]:
def retrieve(question, max_results=5, max_distance=0.7):
    results = collection.query(query_texts=[question], n_results=max_results, include=["documents", "distances"])
    kept = [(doc_id, doc) for doc_id, doc, dist in zip(results["ids"][0], results["documents"][0], results["distances"][0])
            if dist <= max_distance]
    return kept

for q in ["How do organizations verify user identity and secure access?", "What made Stuxnet different from ordinary malware?"]:
    print(f"{q}\n   kept: {[doc_id for doc_id, _ in retrieve(q)]}")


How do organizations verify user identity and secure access?
   kept: ['document_3', 'document_5', 'document_6', 'document_1']


What made Stuxnet different from ordinary malware?
   kept: ['document_2']


## 4. RAG: hand the documents to the model

Retrieval is done; now generation. The prompt has the same three parts as in the previous course: an instruction that says *answer only from the sources below*, the retrieved documents with numbered labels, and the question. Let's print the exact prompt once, because this text is the whole trick.


In [10]:
def build_prompt(question, hits):
    sources = "\n\n".join(f"[{i}] ({doc_id})\n{doc}" for i, (doc_id, doc) in enumerate(hits, start=1))
    return (
        "Answer the question using ONLY the sources below. Cite the sources you use as [1], [2], ...\n"
        "If the sources do not contain the answer, say: I don't have enough information.\n\n"
        f"SOURCES:\n{sources}\n\nQUESTION: {question}"
    )

question = "How does multi-factor authentication protect against attacks?"
hits = retrieve(question, max_results=2)
prompt = build_prompt(question, hits)

print(prompt[:700], "\n[...]")


Answer the question using ONLY the sources below. Cite the sources you use as [1], [2], ...
If the sources do not contain the answer, say: I don't have enough information.

SOURCES:
[1] (document_5)
Multi-Factor Authentication (MFA) is a security mechanism that requires users to provide two or more verification factors to gain access to a resource, such as an application, online account, or VPN. Rather than just asking for a username and password, MFA requires additional credentials, making it harder for attackers to gain unauthorized access. The factors fall into three categories: something you know (like a password or PIN), something you have (like a smartphone or security token), and some 
[...]


In [11]:
answer = client.responses.create(model=MODEL, input=prompt).output_text
print(answer)


Multi-factor authentication (MFA) protects against attacks by requiring two or more verification factors—such as a password, a phone or security token, and biometric data—before granting access. This means that stealing a password alone is not enough for an attacker to access an account. [1]

MFA significantly reduces account takeovers, including those resulting from phishing, and can block over 99.9% of account-compromise attacks, according to Microsoft. [1] In a Zero Trust model, MFA provides the baseline for strong identity verification, with access continuously validated rather than trusted solely because of a user’s network location. [2]

However, SMS-based MFA can be vulnerable to SIM-swapping; authenticator apps and hardware tokens provide more secure alternatives. [1]


### 🔍 What just happened?

The model did not remember anything about MFA. It read the two articles Chroma handed it and wrote an answer from them, with citations pointing at the source labels. Change the articles and the answer changes; that is what "grounded" means. The citations are cheap insurance: a reader can check every claim against a specific document.


Search, prompt, generate: the whole pipeline is one short function. It also prints which documents were used, so you can always check the model's homework.


In [12]:
def rag(question, max_results=3):
    hits = retrieve(question, max_results=max_results)
    answer = client.responses.create(model=MODEL, input=build_prompt(question, hits)).output_text
    print(f"❓ {question}\n   used: {[doc_id for doc_id, _ in hits]}\n💬 {answer}\n")
    return answer

rag("What made Stuxnet different from ordinary malware?")
rag("Why is SMS a weak second factor?");


❓ What made Stuxnet different from ordinary malware?
   used: ['document_2']
💬 Stuxnet differed from ordinary malware because it was a highly sophisticated cyber-physical weapon designed to cause real-world destruction. It targeted Iran’s Siemens industrial control systems and uranium-enrichment centrifuges, crossed an air gap via infected USB drives, exploited multiple zero-day vulnerabilities, concealed its activity by replaying normal operating data, and subtly altered centrifuge speeds until they mechanically failed. [1]



❓ Why is SMS a weak second factor?
   used: ['document_5']
💬 SMS is a weak second factor because it is vulnerable to SIM-swapping attacks, in which attackers convince mobile carriers to transfer a victim’s phone number to a new SIM card and receive the authentication codes. [1]



## 5. When the answer is not in the store

The most important test of any RAG system. Ask about something the documents do not cover and check that the model says so instead of inventing an answer. The instruction in the prompt does most of the work; the distance threshold in `retrieve()` helps too, because an unrelated question brings back nothing at all.


In [13]:
rag("Who is the CEO of Apple and what did the company announce this year?");


❓ Who is the CEO of Apple and what did the company announce this year?
   used: []
💬 I don't have enough information.



### 🔍 What just happened?

Nothing about Apple passed the distance threshold, so the model got an empty source list and, following the instruction, declined to answer. Without the threshold it would have received two unrelated articles and most models would still decline, but some would try to be helpful and guess. Test both paths in your own systems.

### 🎯 Your turn

1. Ask three questions: one answered by a single article, one that needs two articles (hint: phishing and MFA), one the store cannot answer.
2. Change `build_prompt` so the answer must end with a line `Sources used: ...` listing the document ids.
3. Point `retrieve()` at the `cyber_docs_openai` collection from the first notebook (open it with the OpenAI embedding function) and compare the distances. They live on a different scale, so the threshold needs retuning.


In [14]:
# YOUR CODE HERE


## 6. Optional: everything local with an open model

So far retrieval was local and generation went to OpenAI. If the documents must never leave the building, swap the generation step for an open model running on your machine. `Qwen/Qwen2.5-1.5B-Instruct` is small enough for a laptop, and the Hugging Face `transformers` library loads it in two lines. Nothing else in the pipeline changes: same `retrieve()`, same prompt.

On Colab pick a GPU runtime first (**Runtime > Change runtime type > T4 GPU**); on a CPU the first answer takes a minute or two. The model download is about 3 GB.


In [15]:
%pip install -q transformers==5.0.0 torch accelerate

import transformers
from transformers import pipeline
transformers.logging.set_verbosity_error()   # hide the library's generation-config chatter

generate = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", dtype="auto", device_map="auto")
print("model loaded on:", generate.model.device)


Note: you may need to restart the kernel to use updated packages.


model loaded on: mps:0


In [16]:
def local_rag(question, max_results=2, max_new_tokens=150):
    hits = retrieve(question, max_results=max_results)
    context = "\n\n".join(doc for _, doc in hits)
    prompt = f"Answer the question in two or three sentences using the context.\n\nCONTEXT:\n{context}\n\nQUESTION: {question}"
    out = generate([{"role": "user", "content": prompt}], max_new_tokens=max_new_tokens)
    answer = out[0]["generated_text"][-1]["content"]
    print(f"❓ {question}\n   used: {[doc_id for doc_id, _ in hits]}\n💬 {answer}\n")
    return answer

local_rag("What is the difference between authenticator apps and SMS-based MFA?");


❓ What is the difference between authenticator apps and SMS-based MFA?
   used: ['document_5']
💬 Authenticator apps and SMS-based MFA differ in how they generate authentication codes. SMS-based MFA involves sending a one-time password (OTP) via text message to a user's phone. In contrast, authenticator apps like Google Authenticator or Microsoft Authenticator generate a unique code at regular intervals that must be entered along with the initial OTP sent via SMS. This provides an extra layer of security beyond just having access to the phone, as even if someone gains physical control of the phone, they cannot generate valid codes without knowing the secret key associated with the app.



### 🔍 What just happened?

A 1.5-billion-parameter model answered from the same articles, on your hardware, with no API call. The prompt is simpler on purpose: small models follow strict rules like "cite as [1]" and "say you don't know" much less reliably than the hosted model, so the local version asks only for a short answer from the context. The result is usually a little rougher, and the machine did more work, but the data stayed put. Which trade-off is right depends on the documents: for public manuals use the cheap API, for medical records or contracts consider the local route.


## What you take with you

- `query()` embeds the question with the collection's model and returns the nearest documents with **distances**; smaller is closer.
- `where` and `where_document` narrow the search **before** ranking; this is how permissions, dates and exact names get into retrieval.
- **RAG** is retrieve, build a prompt with numbered sources, generate. Citations and a distance threshold are the two cheapest quality tools.
- The generation model is a swappable part: hosted for convenience, local for privacy.

Next notebook: **chunking and overlap**. Every article here was one vector, and the embedding model quietly read only its first half; notebook 3 cuts documents into chunks, shows where chunk vectors land next to the document vector, and tunes chunk size and overlap by experiment. After that: the same ideas in **Pinecone**, a managed vector database in the cloud.
